In [2]:
# 6. Membuat model untuk ke-33 provinsi yang lainnya

import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.metrics import mean_absolute_percentage_error
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

df = pd.read_csv('../data/processed/dataset_lengkap.csv')
df['Date'] = pd.to_datetime(df['Date'])

provinsi = df['Province_Name'].unique()
hasil_evaluasi = []
prediksi_masa_depan = []

for prov in tqdm(provinsi):
    # 6.1. Ambil data per provinsi
    df_prov = df[df['Province_Name'] == prov].copy()
    df_prov = df_prov.sort_values('Date').reset_index(drop=True)
    
    # 6.2. Feature Engineering
    df_prov['Target_Price_7d'] = df_prov['Price'].shift(-7)
    df_prov['DayOfWeek'] = df_prov['Date'].dt.dayofweek
    df_prov['Month'] = df_prov['Date'].dt.month
    
    df_prov['Price_Lag_1'] = df_prov['Price'].shift(1)
    df_prov['Price_Lag_3'] = df_prov['Price'].shift(3)
    df_prov['Price_Roll_Mean_7'] = df_prov['Price'].rolling(window=7).mean()
    df_prov['Price_Roll_Mean_14'] = df_prov['Price'].rolling(window=14).mean()
    
    df_prov['Rain_Roll_Mean_7'] = df_prov['Precipitation'].rolling(window=7).mean()
    df_prov['Temp_Roll_Mean_7'] = df_prov['Temperature'].rolling(window=7).mean()
    
    # 6.3. Pisahkan baris terakhir untuk diprediksi
    data_masa_depan = df_prov.iloc[[-1]].copy() 
    
    # 6.4. Hapus NaN untuk proses training
    df_train_ready = df_prov.dropna().reset_index(drop=True)
    
    features = [
        'Price', 'Price_Lag_1', 'Price_Lag_3', 'Price_Roll_Mean_7', 'Price_Roll_Mean_14',
        'Temperature', 'Precipitation', 'Rain_Roll_Mean_7', 'Temp_Roll_Mean_7',
        'DayOfWeek', 'Month'
    ]
    target = 'Target_Price_7d'
    
    X = df_train_ready[features]
    y = df_train_ready[target]
    
    # Split
    X_train, X_test = X.iloc[:-60], X.iloc[-60:]
    y_train, y_test = y.iloc[:-60], y.iloc[-60:]
    
    # 6.5. Model Training
    model = xgb.XGBRegressor(
        n_estimators=150, learning_rate=0.05, max_depth=5, 
        subsample=0.8, colsample_bytree=0.8, random_state=42, objective='reg:squarederror'
    )
    model.fit(X_train, y_train)
    
    # 6.6. Evaluasi Test Data
    y_pred = model.predict(X_test)
    mape = mean_absolute_percentage_error(y_test, y_pred) * 100
    hasil_evaluasi.append({'Province_Name': prov, 'MAPE_Score': mape})
    
    # 6.7. Prediksi Harga 7 Hari ke Depan
    X_future = data_masa_depan[features]
    future_price = model.predict(X_future)[0]
    
    current_price = data_masa_depan['Price'].values[0]
    
    prediksi_masa_depan.append({
        'Province_Name': prov,
        'Current_Price': current_price,
        'Predicted_Price_7d': future_price,
        'Price_Difference': future_price - current_price
    })

df_evaluasi = pd.DataFrame(hasil_evaluasi)
df_prediksi = pd.DataFrame(prediksi_masa_depan)

# Menghitung Rata-rata MAPE Keseluruhan
mean_mape_nasional = df_evaluasi['MAPE_Score'].mean()

print(f"Rata-rata MAPE Nasional (34 Provinsi): {mean_mape_nasional:.4f}%")

# Simpan hasil prediksi untuk Tahap Optimasi
df_prediksi.to_csv('../data/processed/prediksi_masa_depan.csv', index=False)
print("\nTop 5 Provinsi dengan Prediksi Kenaikan Harga Tertinggi:")
display(df_prediksi.sort_values('Price_Difference', ascending=False).head())
display(df_prediksi.sort_values('Price_Difference', ascending=True).head())

100%|██████████| 34/34 [00:03<00:00, 10.96it/s]

Rata-rata MAPE Nasional (34 Provinsi): 0.8952%

Top 5 Provinsi dengan Prediksi Kenaikan Harga Tertinggi:


,Province_Name,Current_Price,Predicted_Price_7d,Price_Difference
6,Gorontalo,15350.0,16732.246094,1382.246094
28,Sulawesi Tengah,15300.0,15841.603516,541.603516
4,DI Yogyakarta,14200.0,14707.632812,507.632812
10,Jawa Timur,14600.0,14914.694336,314.694336
9,Jawa Tengah,15150.0,15462.404297,312.404297


,Province_Name,Current_Price,Predicted_Price_7d,Price_Difference
31,Sumatera Barat,18300.0,17451.482422,-848.517578
5,DKI Jakarta,16550.0,16263.589844,-286.410156
17,Kepulauan Riau,15450.0,15184.362305,-265.637695
32,Sumatera Selatan,15650.0,15456.507812,-193.492188
23,Papua,19050.0,18857.816406,-192.183594
